# Biohub - Cell Tracking During Development: Quick Baseline Starter

This notebook implements an automated 3D cell centroid detection and nearest-neighbor tracking pipeline:
1. Dynamic discovery of competition test directory.
2. Fast blosc2 volumetric chunk decompression.
3. Spatial downsampling and connected component centroid detection.
4. Bipartite temporal linking via Hungarian matching under physical coordinate constraints.
5. Strict schema verification and generation of .

In [ ]:
import os
import sys
import json
import time
import blosc2
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import label, uniform_filter
from scipy.optimize import linear_sum_assignment

print(f"Python: {sys.version}")
print(f"Numpy: {np.__version__}")
print(f"Blosc2: {blosc2.__version__}")


In [ ]:
# Locate test directory
CANDIDATE_PATHS = [
    "/kaggle/input/biohub-cell-tracking-during-development/test",
    "/kaggle/input/competitions/biohub-cell-tracking-during-development/test"
]

TEST_DIR = None
for path_str in CANDIDATE_PATHS:
    p = Path(path_str)
    if p.exists():
        TEST_DIR = p
        break

if TEST_DIR is None:
    raise FileNotFoundError(f"Test directory not found in candidates: {CANDIDATE_PATHS}")

print(f"Active Test Directory: {TEST_DIR}")
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X physical voxel scale (um)
DOWNSAMPLE = 4
PERCENTILE = 90
MAX_LINK_DISTANCE = 15.0  # um


In [ ]:
test_folders = sorted(
    d.name.replace(".zarr", "") for d in TEST_DIR.glob("*.zarr")
)
print(f"Found {len(test_folders)} test volumes: {test_folders}")

start_total = time.time()
all_rows = []
global_id = 0

for folder_idx, folder_name in enumerate(test_folders, start=1):
    t0 = time.time()
    zarr_path = TEST_DIR / f"{folder_name}.zarr"
    meta_file = zarr_path / "0" / "zarr.json"
    with open(meta_file, "r") as f:
        meta = json.load(f)
    
    shape = tuple(meta["shape"])  # (T, Z, Y, X)
    dtype = np.dtype(meta["data_type"])
    n_t = shape[0]
    
    prev_centroids = {}
    node_id_counter = 1
    nodes_count = 0
    edges_count = 0
    
    for t in range(n_t):
        chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
        with open(chunk_path, "rb") as f:
            compressed = f.read()
        decomp = blosc2.decompress(compressed)
        vol = np.frombuffer(decomp, dtype=dtype).reshape(shape[1:])
        
        ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
        smoothed = uniform_filter(ds.astype(np.float32), size=3)
        th = np.percentile(smoothed, PERCENTILE)
        binary = smoothed > th
        labeled, n_features = label(binary)
        
        curr_centroids = {}
        for comp_id in range(1, n_features + 1):
            coords = np.argwhere(labeled == comp_id)
            centroid = coords.mean(axis=0) * DOWNSAMPLE
            nid = node_id_counter
            node_id_counter += 1
            cz = int(round(centroid[0]))
            cy = int(round(centroid[1]))
            cx = int(round(centroid[2]))
            curr_centroids[nid] = (cz, cy, cx)
            
            all_rows.append({
                "id": global_id,
                "dataset": folder_name,
                "row_type": "node",
                "node_id": nid,
                "t": t,
                "z": cz,
                "y": cy,
                "x": cx,
                "source_id": -1,
                "target_id": -1
            })
            global_id += 1
            nodes_count += 1
            
        # Bipartite matching for edge formation
        if prev_centroids and curr_centroids:
            p_ids = list(prev_centroids.keys())
            c_ids = list(curr_centroids.keys())
            p_coords = np.array([prev_centroids[pid] for pid in p_ids], dtype=np.float64)
            c_coords = np.array([curr_centroids[cid] for cid in c_ids], dtype=np.float64)
            
            p_phys = p_coords * SCALE
            c_phys = c_coords * SCALE
            
            dist = np.sqrt(((p_phys[:, None] - c_phys[None, :]) ** 2).sum(axis=2))
            row_ind, col_ind = linear_sum_assignment(dist)
            
            for ri, ci in zip(row_ind, col_ind):
                if dist[ri, ci] <= MAX_LINK_DISTANCE:
                    all_rows.append({
                        "id": global_id,
                        "dataset": folder_name,
                        "row_type": "edge",
                        "node_id": -1,
                        "t": -1,
                        "z": -1,
                        "y": -1,
                        "x": -1,
                        "source_id": p_ids[ri],
                        "target_id": c_ids[ci]
                    })
                    global_id += 1
                    edges_count += 1
                    
        prev_centroids = curr_centroids
    
    elapsed = time.time() - t0
    print(f"[{folder_idx}/{len(test_folders)}] {folder_name} finished in {elapsed:.1f}s ({nodes_count} nodes, {edges_count} edges)")

print(f"All inference finished in {time.time() - start_total:.1f}s. Total rows: {len(all_rows)}")


In [ ]:
# Save and validate submission.csv
df = pd.DataFrame(all_rows)
cols = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
df = df[cols]
out_path = Path("submission.csv")
df.to_csv(out_path, index=False)

print(f"Written {len(df)} rows to {out_path.resolve()}")
print(f"File size: {out_path.stat().st_size / 1024:.2f} KB")
print("Preview:")
print(df.head(10))
print(df.tail(10))

# Verification checks
assert out_path.exists(), "submission.csv does not exist!"
assert len(df) > 0, "submission.csv is empty!"
for fn in test_folders:
    assert fn in df["dataset"].values, f"Dataset {fn} missing from submission!"
print("All verification checks passed successfully!")
